In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
DIFFUSION_CONFIG_PATH = "config/diffusion_full_celeba.yaml"   # relative to repo root
DIFFUSION_CKPT_PATH = None   # None → load latest checkpoint from experiment output dir
N_SAMPLES = 4
SAMPLER = "ddim"  # 'ddpm' or 'ddim'
NUM_INFERENCE_STEPS = 50     # ddim only: steps to run (<T is faster); ignored for ddpm
ETA = 0.0    # ddim only: 0=deterministic, 1≈ddpm stochasticity
GUIDANCE_SCALE = 1.0    # >1.0 requires a CFG-trained model (cfg_uncond_prob > 0)
SEED = 42

import random, torch
random.seed(SEED)
torch.manual_seed(SEED)

## Imports

In [ ]:
import os
import sys
import types
import glob
import yaml

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from models.beta_vae import BetaVAE
from models.unet import UNet
from data_utils import build_dataloaders
from utils import make_noise_schedule, diffusion_sample, make_run_tag

## Load models

In [ ]:
with open(os.path.join(REPO_ROOT, DIFFUSION_CONFIG_PATH)) as f:
    diff_cfg = types.SimpleNamespace(**yaml.safe_load(f))

diff_cfg.output_dir          = os.path.join(REPO_ROOT, f"{diff_cfg.output_dir.lstrip('./')}_{make_run_tag(diff_cfg)}")
diff_cfg.betavae_config_path = os.path.join(REPO_ROOT, diff_cfg.betavae_config_path.lstrip("./"))
diff_cfg.betavae_ckpt_path   = os.path.join(REPO_ROOT, diff_cfg.betavae_ckpt_path.lstrip("./"))
if not os.path.isabs(diff_cfg.data_dir):
    diff_cfg.data_dir = os.path.join(REPO_ROOT, diff_cfg.data_dir.lstrip("./"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- frozen β-VAE ---
with open(diff_cfg.betavae_config_path) as f:
    vae_cfg = types.SimpleNamespace(**yaml.safe_load(f))

vae = BetaVAE(
    input_channels=vae_cfg.input_channels,
    input_height=vae_cfg.input_height,
    input_width=vae_cfg.input_width,
    latent_dim=vae_cfg.latent_dim,
    hidden_dim=vae_cfg.hidden_dim,
    beta=vae_cfg.beta,
    device=device,
)
vae_ckpt = torch.load(diff_cfg.betavae_ckpt_path, map_location=device)
vae.load_state_dict(vae_ckpt["model_state_dict"])
vae.eval()
print(f"beta-VAE loaded (step {vae_ckpt['step']})")

# --- UNet ---
unet = UNet(
    in_channels=diff_cfg.in_channels,
    image_size=diff_cfg.image_size,
    model_channels=diff_cfg.model_channels,
    channel_mult=diff_cfg.channel_mult,
    num_res_blocks=diff_cfg.num_res_blocks,
    attention_resolutions=diff_cfg.attention_resolutions,
    dropout=0.0,
    latent_dim=diff_cfg.latent_dim,
)

if DIFFUSION_CKPT_PATH is None:
    ckpt_dir = os.path.join(diff_cfg.output_dir, "checkpoints")
    DIFFUSION_CKPT_PATH = os.path.join(ckpt_dir, "best_ckpt.pt")
    if not os.path.exists(DIFFUSION_CKPT_PATH):
        raise FileNotFoundError(f"best_ckpt.pt not found in {ckpt_dir}")

diff_ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location=device)
unet.load_state_dict(diff_ckpt["model_state_dict"])
unet = unet.to(device).eval()
print(f"UNet loaded: {DIFFUSION_CKPT_PATH}  (step {diff_ckpt['step']})")

## Noise schedule

In [ ]:
schedule = make_noise_schedule(diff_cfg, device)
T = diff_cfg.num_timesteps
print(f"Schedule ready: T={T}, beta_0={schedule['betas'][0]:.5f}, beta_T={schedule['betas'][-1]:.4f}")

## Sampling

`diffusion_sample` is imported from `utils.py` — see there for full docstring.
Switching between DDPM and DDIM, CFG, verbose mode, and accelerated DDIM steps are all controlled by the config cell above.

## Sample images and generate

In [ ]:
_, val_dl = build_dataloaders(diff_cfg)
all_images = torch.cat([b for b in val_dl], dim=0)

import random
indices   = random.sample(range(len(all_images)), min(N_SAMPLES, len(all_images)))
originals = all_images[indices].to(device)

with torch.no_grad():
    z, _ = vae.encode(originals)
    vae_recon = torch.sigmoid(vae.decode(z)).cpu()

generated = diffusion_sample(
    z, unet, schedule, diff_cfg, device,
    sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
).cpu()
originals_cpu = originals.cpu()

## Visualize

In [ ]:
n = len(indices)
fig, axes = plt.subplots(3, n, figsize=(n * 2.5, 7))
fig.suptitle("original  |  β-VAE reconstruction  |  diffusion generated", fontsize=11)

rows = [originals_cpu, vae_recon, generated]
labels = ["original", "β-VAE recon", "diffusion"]

for row_idx, (imgs, label) in enumerate(zip(rows, labels)):
    axes[row_idx, 0].set_ylabel(label, fontsize=9)
    for col_idx in range(n):
        axes[row_idx, col_idx].imshow(imgs[col_idx].permute(1, 2, 0).clamp(0, 1))
        axes[row_idx, col_idx].axis("off")

plt.tight_layout()
plt.show()

## Denoising progression (verbose)

In [ ]:
z_single = z[:1]
_, snapshots = diffusion_sample(
    z_single, unet, schedule, diff_cfg, device,
    sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    verbose=True, n_snapshots=10,
)

n_snaps = len(snapshots)
fig, axes = plt.subplots(1, n_snaps, figsize=(n_snaps * 2, 2.8))
fig.suptitle(
    f"{SAMPLER.upper()} progression  (left: noise  →  right: final)"
    + (f"  [{NUM_INFERENCE_STEPS} steps]" if SAMPLER == 'ddim' else "  [1000 steps]"),
    fontsize=11,
)
for ax, (t_idx, img) in zip(axes, snapshots):
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"t={t_idx}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()